### Data ingestion incremental

In [0]:
dbutils.widgets.text("src","")

In [0]:
src_value = dbutils.widgets.get("src")
src_value

In [0]:
from pyspark.sql.functions import *

In [0]:
df = (spark.readStream.format("cloudFiles")\
        .option("cloudFiles.format", "csv")\
        .option("cloudFiles.schemaLocation",f"/Volumes/workspace/bronze/bronzevolume/{src_value}/schema")\
        .load(f"/Volumes/workspace/raw_datavricksproject/rawvolume/rawdata/{src_value}/")\
        .withColumn("insertedDate", current_timestamp())
)


In [0]:
query = (
    df.writeStream
    .format("delta")
    .outputMode("append")
    .trigger(once=True)
    .option("mergeSchema", "true")
    .option("checkpointLocation", f"/Volumes/workspace/bronze/bronzevolume/{src_value}/checkpoint_V2")
    .option("path", f"/Volumes/workspace/bronze/bronzevolume/{src_value}/data")
    .start()
)

query.awaitTermination()